# Setup

In [ ]:
from collections.abc import Callable
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import numpy.typing as npt
import polars as pl
import seaborn as sns
from tqdm import trange

from climate_attitudes.dataset import Dataset
from climate_attitudes.settings import Config
from climate_attitudes.visualisation import configure_mpl
from ising import Ising

configure_mpl(Path("../fonts/"))

np.set_printoptions(linewidth=200)

RANDOM_SEED = 202606081503

In [ ]:
config = Config(_env_file="../.env")
dataset = Dataset.load(
    config,
    name="reduced_no_imputation",
    with_imputation=False,
    verbose=False,
)
_, Y, _ = dataset.indices_to_numpy(kind="time-series", binarise=True, seed=RANDOM_SEED)

In [ ]:
model = Ising.fit(Y, update_method="sync", rng=RANDOM_SEED)
imodel = model.intervene(
    spins=np.array([2]), field_offset=np.array([0.5]), seed=RANDOM_SEED
)
model.reset(RANDOM_SEED)
imodel.reset(RANDOM_SEED)

# Shapley value calculation

We calculate shapley values for one individual ($i = 0$), for the task of predicting the next state of the spin at index 5. Our features are the initial spin states.

In [ ]:
y = Y[2]
y

To do so, we need to be able to calculate the expected value of this state, conditional on us only knowing the initial states of some spins, i.e., those in the set $Q$:

$$
\begin{align}
f_Q(\boldsymbol{s}) &= \mathbb{E}[f\mid S_i = s_i, \forall i \in Q] \\
&= \sum_{\boldsymbol{s}_{-Q}} P(\boldsymbol{s}_{-Q}) \cdot f(s')
\end{align}
$$

Where $s'$ comprises the values from $\boldsymbol{S}$ for those spins in $Q$, and $\boldsymbol{s}_{-Q}$ for all others.

While we don't know the values of the probabilities in the summation, we can approximate this value by drawing repeated samples based on the _previous_ timestep's state, and fixing the spins in $Q$ to their assigned values.

In [ ]:
def int_effect_1(model: Ising, imodel: Ising, y: npt.NDArray[np.int64]) -> np.int64:
    y_null = y
    y_int = y
    for _ in range(5):
        y_null = model.step_parallel(y_null)
        y_int = imodel.step_parallel(y_int)

    # print(y_null[1], y_int[1])

    return (y_int - y_null)[1]


def s1(model: Ising, y: npt.NDArray[np.int64]) -> np.int64:
    y_next = model.step_parallel(y)
    return y_next[1]


def s5(model: Ising, y: npt.NDArray[np.int64]) -> np.int64:
    y_next = model.step_parallel(y)
    return y_next[5]


def conditional_expectation[T: np.number](
    f: Callable[[Ising, npt.NDArray[np.int64]], T],
    y: npt.NDArray[np.int64],
    y_prev: npt.NDArray[np.int64],
    do: npt.NDArray[np.int64],
    model: Ising,
    imodel: Ising,
    samples: int = 100,
) -> np.float64:
    inits = np.empty((samples, y_prev.size), dtype=np.int64)
    for i in range(samples):
        inits[i] = model.step_parallel(y_prev)
        imodel.step_parallel(y_prev)
        inits[i][do] = y[do]

    evals = np.empty(samples, dtype=np.float64)
    for i, y_init in enumerate(inits):
        evals[i] = f(model, imodel, y_init)

    return evals.mean()

In [ ]:
conditional_expectation(
    int_effect_1,
    y[1],
    y[0],
    do=np.array([1, 0]),
    model=model,
    imodel=imodel,
    samples=10_000,
)

In [ ]:
np.arange(12)[:5]

The next step is to define the contribution of a particular subset of features $Q$, which is:

$$\Delta_Q(\boldsymbol{s}) = f_Q(\boldsymbol{s}) - f_{\{\}}(\boldsymbol{s})$$

In [ ]:
def marginal_contribution[T: np.number](
    f: Callable[[Ising, npt.NDArray[np.int64]], T],
    y: npt.NDArray[np.int64],
    y_prev: npt.NDArray[np.int64],
    i: int | np.int64,
    pi: npt.NDArray[np.int64],
    # do: npt.NDArray[np.int64],
    model: Ising,
    imodel: Ising,
    samples: int = 100,
) -> np.float64:
    loc = np.where(pi == i)[0][0]
    b1 = conditional_expectation(f, y, y_prev, pi[: loc + 1], model, imodel, samples)
    b2 = conditional_expectation(f, y, y_prev, pi[:loc], model, imodel, samples)
    return b1 - b2

In [ ]:
marginal_contribution(
    int_effect_1,
    y[1],
    y[0],
    i=3,
    pi=np.array([7, 0, 1, 4, 3, 5, 6, 2]),
    model=model,
    imodel=imodel,
    samples=1_000,
)

In [ ]:
def spin_shapley_approx[T: np.number](
    f: Callable[[Ising, npt.NDArray[np.int64]], T],
    i: int,
    y: npt.NDArray[np.int64],
    y_prev: npt.NDArray[np.int64],
    model: Ising,
    imodel: Ising,
    rng: np.random.Generator,
    model_samples: int = 100,
    shapley_samples: int = 100,
) -> np.float64:
    acc = np.float64(0.0)
    n = y.size
    for _ in range(shapley_samples):
        pi = rng.permutation(n)
        acc += marginal_contribution(f, y, y_prev, i, pi, model, imodel, model_samples)
    return acc / shapley_samples


def shapley_approx[T: np.number](
    f: Callable[[Ising, npt.NDArray[np.int64]], T],
    y: npt.NDArray[np.int64],
    y_prev: npt.NDArray[np.int64],
    model: Ising,
    imodel: Ising,
    rng: np.random.Generator,
    model_samples: int = 100,
    shapley_samples: int = 100,
) -> npt.NDArray[np.float64]:
    evals = np.empty_like(y, dtype=np.float64)
    for i in range(y.size):
        evals[i] = spin_shapley_approx(
            f, i, y, y_prev, model, imodel, rng, model_samples, shapley_samples
        )
    return evals

In [ ]:
Y[1567, 1]

In [ ]:
m = 1568
model_samples = 100
shapley_samples = 30
rng = np.random.default_rng(RANDOM_SEED + 1)

shap_vals = np.empty((1, 8), dtype=np.float64)

for i in trange(1567, m):
    shap_vals[i - 1567] = shapley_approx(
        int_effect_1,
        Y[i][1],
        Y[i][0],
        model=model,
        imodel=imodel,
        rng=rng,
        model_samples=model_samples,
        shapley_samples=shapley_samples,
    )

In [ ]:
shap_vals

In [ ]:
cols = [
    "Belief CC",
    "CC Anthro",
    "CC Worry",
    "CC Worry (others)",
    "Weather worry",
    "Politics",
    "Climate impacts",
    "Climate policy",
]
plot_df = pl.DataFrame(
    {
        "Spin": [c for _ in range(1) for c in cols],
        "Shapley value": shap_vals.flatten(),
        # "State": shap_values.data.flatten(),
    }
)

In [ ]:
fig, ax = plt.subplots(constrained_layout=True)
sns.stripplot(plot_df, y="Spin", x="Shapley value", orient="h", s=3, ax=ax)

ax.set_yticks(np.arange(8), cols)
ax.set_xlim(-0.035, 0.035)

In [ ]:
repeats = 30
shaps = np.empty(repeats, dtype=np.float64)

for r in trange(repeats):
    shaps[r] = spin_shapley_approx(
        int_effect_1,
        1,
        Y[1567, 1],
        Y[1567, 0],
        model=model,
        imodel=imodel,
        rng=rng,
        model_samples=100,
        shapley_samples=30,
    )

In [ ]:
shaps.mean()

In [ ]:
model.reset(RANDOM_SEED + 1)
imodel.reset(RANDOM_SEED + 1)

null_outcome = model.measure(s1, y0=Y[:, 1], t=6, repeats=100, warmup_steps=0).y[
    ..., -1
]
int_outcome = imodel.measure(s1, y0=Y[:, 1], t=6, repeats=100, warmup_steps=0).y[
    ..., -1
]

In [ ]:
null_outcome.shape

In [ ]:
int_effect = (int_outcome - null_outcome).mean(axis=1)

In [ ]:
np.argwhere(int_effect == int_effect.max())

In [ ]:
int_effect[1567]

In [ ]:
Y[1567, -1]

In [ ]:
sns.displot(int_effect)

In [ ]:
conditional_expectation(
    int_effect_1,
    Y[1567, 1],
    Y[1567, 0],
    do=np.array([], dtype=np.int64),
    model=model,
    imodel=imodel,
    samples=10_000,
)